# BUAN381 Final Project — AI Adoption & Revenue Growth Prediction Analysis

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Data Loading

In [ ]:
# Colab-compatible path detection
if os.path.exists('../dataset/ai_company_adoption.csv'):
    DATA_DIR = '../dataset'
    FIG_DIR = '../report'
elif os.path.exists('dataset/ai_company_adoption.csv'):
    DATA_DIR = 'dataset'
    FIG_DIR = 'figures'
else:
    os.system('git clone https://github.com/Braden357/buan390-final.git')
    DATA_DIR = 'buan390-final/dataset'
    FIG_DIR = 'buan390-final/figures'

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print(f"DATA_DIR: {DATA_DIR}  |  FIG_DIR: {FIG_DIR}")

df = pd.read_csv(f'{DATA_DIR}/ai_company_adoption.csv')
print(f"Shape: {df.shape}")
print(f"Nulls: {df.isnull().sum().sum()}")
df.head(3)

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['revenue_growth_percent'], bins=50, edgecolor='black', color='steelblue')
axes[0].set_title('Revenue Growth % Distribution')
axes[0].set_xlabel('Revenue Growth (%)')
axes[0].set_ylabel('Count')

axes[1].boxplot(df['revenue_growth_percent'])
axes[1].set_title('Revenue Growth % Boxplot')
axes[1].set_ylabel('Revenue Growth (%)')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_target_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(df['revenue_growth_percent'].describe().round(2))

In [ ]:
industry_avg = df.groupby('industry')['revenue_growth_percent'].agg(['mean','std','count']).reset_index()
industry_avg.columns = ['industry','mean_revenue_growth','std_revenue_growth','count']
industry_avg = industry_avg.sort_values('mean_revenue_growth', ascending=False)

plt.figure(figsize=(12, 6))
bars = plt.bar(industry_avg['industry'], industry_avg['mean_revenue_growth'], color='steelblue', edgecolor='black')
plt.title('Average Revenue Growth by Industry', fontsize=14, fontweight='bold')
plt.xlabel('Industry')
plt.ylabel('Mean Revenue Growth (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_revenue_by_industry.png', dpi=150, bbox_inches='tight')
plt.show()

industry_avg.to_csv(f'{DATA_DIR}/tableau_industry_summary.csv', index=False)

In [ ]:
num_cols = ['ai_adoption_rate','ai_maturity_score','ai_budget_percentage','ai_training_hours',
            'num_ai_tools_used','ai_projects_active','task_automation_rate','time_saved_per_week',
            'ai_investment_per_employee','num_employees','annual_revenue_usd_millions',
            'company_age','revenue_growth_percent']

corr = df[num_cols].corr()
plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Matrix — Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
size_order = ['Startup', 'SME', 'Enterprise']
size_data = [df[df['company_size'] == s]['revenue_growth_percent'].values for s in size_order]

plt.figure(figsize=(10, 6))
plt.boxplot(size_data, tick_labels=size_order)
plt.title('Revenue Growth by Company Size', fontsize=14, fontweight='bold')
plt.xlabel('Company Size')
plt.ylabel('Revenue Growth (%)')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_revenue_by_size.png', dpi=150, bbox_inches='tight')
plt.show()

for size in size_order:
    mean = df[df['company_size'] == size]['revenue_growth_percent'].mean()
    print(f"{size}: {mean:.2f}%")

In [ ]:
# For Tableau scatter: AI budget vs revenue growth (sample 5000 rows for performance)
tableau_scatter = df[['ai_budget_percentage','revenue_growth_percent','industry',
                       'company_size','region','ai_adoption_stage']].sample(5000, random_state=42)
tableau_scatter.to_csv(f'{DATA_DIR}/tableau_scatter.csv', index=False)

# For Tableau map: region averages
region_avg = df.groupby('region').agg(
    mean_revenue_growth=('revenue_growth_percent','mean'),
    mean_ai_maturity=('ai_maturity_score','mean'),
    mean_ai_budget=('ai_budget_percentage','mean'),
    count=('revenue_growth_percent','count')
).reset_index()
region_avg.to_csv(f'{DATA_DIR}/tableau_region_summary.csv', index=False)
print("Tableau exports done")

## 3. Feature Engineering

In [ ]:
DROP_COLS = ['response_id','company_id','survey_source','data_collection_method',
             'country','quarter']

df_model = df.drop(columns=DROP_COLS).copy()

CAT_COLS = ['region','industry','company_size','company_age_group','ai_adoption_stage',
            'ai_primary_tool','ai_use_case','data_privacy_level','ai_ethics_committee']

le = LabelEncoder()
for col in CAT_COLS:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

print("Shape after encoding:", df_model.shape)
print("Dtypes:\n", df_model.dtypes.value_counts())

In [ ]:
TARGET = 'revenue_growth_percent'
FEATURES = [c for c in df_model.columns if c != TARGET]

X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 4. Linear Regression (Parametric Model)

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("=== Linear Regression Results ===")
print(f"RMSE: {rmse_lr:.4f}")
print(f"MAE:  {mae_lr:.4f}")
print(f"R²:   {r2_lr:.4f}")

In [ ]:
coef_df = pd.DataFrame({
    'feature': FEATURES,
    'coefficient': lr.coef_
}).sort_values('coefficient', key=abs, ascending=False).head(15)

plt.figure(figsize=(12, 6))
colors = ['steelblue' if c > 0 else 'tomato' for c in coef_df['coefficient']]
plt.barh(coef_df['feature'], coef_df['coefficient'], color=colors, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Linear Regression — Top 15 Feature Coefficients', fontsize=14, fontweight='bold')
plt.xlabel('Coefficient Value')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

print(coef_df.to_string(index=False))

In [ ]:
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(y_test), 2000, replace=False)
y_test_arr = y_test.values
plt.figure(figsize=(8, 8))
plt.scatter(y_test_arr[sample_idx], y_pred_lr[sample_idx], alpha=0.3, color='steelblue', s=10)
plt.plot([y_test_arr.min(), y_test_arr.max()],
         [y_test_arr.min(), y_test_arr.max()], 'r--', linewidth=2, label='Perfect fit')
plt.title('Linear Regression — Actual vs Predicted (sample n=2000)', fontsize=14, fontweight='bold')
plt.xlabel('Actual Revenue Growth (%)')
plt.ylabel('Predicted Revenue Growth (%)')
plt.legend()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_lr_actual_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
cv_scores_lr = cross_val_score(lr, X_train, y_train, cv=5, scoring='r2')
print(f"LR 5-fold CV R²: {cv_scores_lr.mean():.4f} ± {cv_scores_lr.std():.4f}")

## 5. Random Forest — Baseline

In [ ]:
rf_base = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_base.fit(X_train, y_train)
y_pred_rf_base = rf_base.predict(X_test)

rmse_rf_base = np.sqrt(mean_squared_error(y_test, y_pred_rf_base))
mae_rf_base = mean_absolute_error(y_test, y_pred_rf_base)
r2_rf_base = r2_score(y_test, y_pred_rf_base)

print("=== Random Forest (Baseline) ===")
print(f"RMSE: {rmse_rf_base:.4f}")
print(f"MAE:  {mae_rf_base:.4f}")
print(f"R²:   {r2_rf_base:.4f}")

## 6. Random Forest — Hyperparameter Tuning

In [ ]:
# Tune on a 25k-row sample so this finishes in ~5 min on Colab free tier.
# Best hyperparameters are then refit on the full training set before test evaluation.
X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train, train_size=25000, random_state=42)

param_dist = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [2, 5],
    'max_features': ['sqrt', 'log2']
}

rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=2),
    param_distributions=param_dist,
    n_iter=12,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=2,
    verbose=1
)

rf_search.fit(X_tune, y_tune)
print("Best params:", rf_search.best_params_)
print("Best CV R² (tune sample):", round(rf_search.best_score_, 4))

In [ ]:
# Refit best hyperparameters on the full training set, then evaluate on the held-out test set
rf_best = RandomForestRegressor(**rf_search.best_params_, random_state=42, n_jobs=-1)
rf_best.fit(X_train, y_train)
y_pred_rf = rf_best.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("=== Random Forest (Tuned, refit on full train) ===")
print(f"RMSE: {rmse_rf:.4f}")
print(f"MAE:  {mae_rf:.4f}")
print(f"R²:   {r2_rf:.4f}")

feat_imp = pd.DataFrame({
    'feature': FEATURES,
    'importance': rf_best.feature_importances_
}).sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(12, 6))
plt.barh(feat_imp['feature'], feat_imp['importance'], color='steelblue', edgecolor='black')
plt.title('Random Forest — Top 15 Feature Importances', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_rf_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest (Baseline)', 'Random Forest (Tuned)'],
    'RMSE': [rmse_lr, rmse_rf_base, rmse_rf],
    'MAE': [mae_lr, mae_rf_base, mae_rf],
    'R²': [r2_lr, r2_rf_base, r2_rf]
}).round(4)

print("=== Model Comparison ===")
print(results.to_string(index=False))
results

In [ ]:
metrics = ['RMSE', 'MAE', 'R²']
models = ['Linear Regression', 'RF Baseline', 'RF Tuned']
values = {
    'RMSE': [rmse_lr, rmse_rf_base, rmse_rf],
    'MAE': [mae_lr, mae_rf_base, mae_rf],
    'R²': [r2_lr, r2_rf_base, r2_rf]
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['steelblue', 'darkorange', 'seagreen']

for i, metric in enumerate(metrics):
    bars = axes[i].bar(models, values[metric], color=colors, edgecolor='black')
    axes[i].set_title(f'{metric} Comparison ({"lower" if metric != "R²" else "higher"} is better)',
                      fontsize=12, fontweight='bold')
    axes[i].set_ylabel(metric)
    axes[i].set_xticks(range(len(models)))
    axes[i].set_xticklabels(models, rotation=15, ha='right')
    for bar, val in zip(bars, values[metric]):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Model Performance Comparison — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")
print(f"LR: RMSE={rmse_lr:.4f}, MAE={mae_lr:.4f}, R²={r2_lr:.4f}")
print(f"RF Base: RMSE={rmse_rf_base:.4f}, MAE={mae_rf_base:.4f}, R²={r2_rf_base:.4f}")
print(f"RF Tuned: RMSE={rmse_rf:.4f}, MAE={mae_rf:.4f}, R²={r2_rf:.4f}")


## 8. Business Interpretation

### Key Findings

**Linear Regression (R² ≈ 0.24):**
- `ai_budget_percentage` and `ai_maturity_score` are the strongest positive predictors of revenue growth
- Each percentage point increase in AI budget allocation correlates with measurable revenue growth improvement
- The model's moderate R² indicates AI investment explains roughly 24% of revenue growth variance — other factors (market conditions, management quality) account for the rest

**Random Forest — Tuned (R² ≈ 0.23):**
- Non-linear patterns exist but are modest — the RF model does not significantly outperform linear regression
- `ai_maturity_score` and `ai_adoption_rate` dominate feature importance, consistent with LR coefficients
- The similar R² across both models suggests the relationship between AI investment and revenue growth is largely linear

### Strategic Implications

| Company Profile | Recommendation |
|---|---|
| Low AI maturity, high budget | Prioritize maturity before spending — budget alone doesn't predict growth |
| High AI maturity, low budget | High ROI opportunity — increase investment to capitalize on existing capability |
| Enterprise, any stage | Automation rate and training hours matter most — invest in workforce enablement |
| Startup | Focus on AI adoption rate first; early-stage firms see outsized returns from initial adoption |

### Limitations
- Survey data introduces self-reporting bias in AI metrics
- Cross-sectional design — cannot establish causality between AI investment and revenue growth
- R² ~0.24 means most revenue variation is unexplained by AI factors alone